In [1]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
import pandas as pd
from sklearn.utils import shuffle
import os
import numpy as np
os.chdir("../../data/input_datasets")

In [2]:
mimic_path = "../embeddings/MIMIC/embeddings"
mimic_targets = pd.read_csv("mimic_targets.csv", index_col=0)
mimic_patients_df = pd.read_csv("mimic_with_targets_patientIDs.csv")
groups = np.array(mimic_patients_df["subject_id"])
patient_admission_dict = {patient : [index for index, pat in enumerate(groups) if pat==patient] for patient in groups}

In [7]:
NUM_DIMENSION = 64
result_dict = {"Method" : [], "Target" : [], "Type" : [], "Score": [], "Precision": [], "Recall": [], "F1": []}

for i in range(10):
    # Load embeddings of all three datasets.
    mimic_file = os.path.join(mimic_path, f"MIMIC_samples_{NUM_DIMENSION}_{i}.tsv")
    mimic_umap_file = os.path.join(mimic_path, f"MIMIC_UMAP_{NUM_DIMENSION}_{i}.csv")
    
    mimic_df = pd.read_csv(mimic_file, sep='\t', index_col=0)
    mimic_umap = pd.read_csv(mimic_umap_file, index_col=0)
    
    mimic_df = mimic_df.join(mimic_targets)
    mimic_umap = mimic_umap.join(mimic_targets)

    # Init regression and CV models.
    clf = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=1000
    )    

    # Separability analysis for Aplasia.
    mimic_aplasia_df = mimic_df.copy()
    mimic_aplasia_y = mimic_aplasia_df["label_aplasia"].values
    mimic_umap_aplasia_df = mimic_umap.copy()

    pos_class_ratio = mimic_aplasia_y.mean()
    
    patients = np.array(list(patient_admission_dict.keys()))
    patients = shuffle(patients, random_state=42)

    n_folds = 10
    patient_folds = np.array_split(patients, n_folds)

    pome_aplasia_auc_scores = []
    pome_aplasia_precision = []
    pome_aplasia_recall = []
    pome_aplasia_f1 = []
    umap_aplasia_auc_scores = []
    umap_aplasia_precision = []
    umap_aplasia_recall = []
    umap_aplasia_f1 = []
    
    for fold_idx in range(n_folds):

        # Patients in test fold
        test_patients = patient_folds[fold_idx]

        # Patients in train folds
        train_patients = np.concatenate(
            [patient_folds[i] for i in range(n_folds) if i != fold_idx]
        )

        # Collect row indices
        train_rows = []
        for p in train_patients:
            train_rows.extend(patient_admission_dict[p])

        test_rows = []
        for p in test_patients:
            test_rows.extend(patient_admission_dict[p])

        # Extract dataframe rows
        pome_train_df = mimic_aplasia_df.iloc[train_rows]
        pome_test_df = mimic_aplasia_df.iloc[test_rows]
        
        umap_train_df = mimic_umap_aplasia_df.iloc[train_rows]
        umap_test_df = mimic_umap_aplasia_df.iloc[test_rows]

        # Compute POME AP.
        pome_X_train = pome_train_df.iloc[:, :NUM_DIMENSION].values
        pome_y_train = pome_train_df["label_aplasia"].values

        pome_X_test = pome_test_df.iloc[:, :NUM_DIMENSION].values
        pome_y_test = pome_test_df["label_aplasia"].values

        clf.fit(pome_X_train, pome_y_train)

        pome_y_pred = clf.predict_proba(pome_X_test)[:, 1]
        
        # Compute precision and recall.
        pome_y_pred_cut = (pome_y_pred >= 0.5).astype(int)
        pome_precision = precision_score(pome_y_test, pome_y_pred_cut)
        pome_recall = recall_score(pome_y_test, pome_y_pred_cut)
        pome_f1 = f1_score(pome_y_test, pome_y_pred_cut)
        pome_score = average_precision_score(pome_y_test, pome_y_pred)
        
        pome_aplasia_auc_scores.append(pome_score)
        pome_aplasia_precision.append(pome_precision)
        pome_aplasia_recall.append(pome_recall)
        pome_aplasia_f1.append(pome_f1)
        
        # Compute UMAP AP.
        assert not "label_aplasia" in umap_train_df.columns[:NUM_DIMENSION]
        umap_X_train = umap_train_df.iloc[:, :NUM_DIMENSION].values
        umap_y_train = umap_train_df["label_aplasia"].values

        assert not "label_aplasia" in umap_test_df.columns[:NUM_DIMENSION]
        umap_X_test = umap_test_df.iloc[:, :NUM_DIMENSION].values
        umap_y_test = umap_test_df["label_aplasia"].values

        clf.fit(umap_X_train, umap_y_train)

        umap_y_pred = clf.predict_proba(umap_X_test)[:, 1]
        # Compute additional precision and recall.
        umap_y_pred_cut = (umap_y_pred >= 0.5).astype(int)
        umap_precision = precision_score(umap_y_test, umap_y_pred_cut)
        umap_recall = recall_score(umap_y_test, umap_y_pred_cut)
        umap_f1 = f1_score(umap_y_test, umap_y_pred_cut)
        umap_score = average_precision_score(umap_y_test, umap_y_pred)
        
        umap_aplasia_auc_scores.append(umap_score)
        umap_aplasia_precision.append(umap_precision)
        umap_aplasia_recall.append(umap_recall)
        umap_aplasia_f1.append(umap_f1)
    
    
    assert len(pome_aplasia_auc_scores)==len(umap_aplasia_auc_scores)
    for i in range(len(pome_aplasia_auc_scores)):
        pome_auc = pome_aplasia_auc_scores[i]
        umap_auc = umap_aplasia_auc_scores[i]
        result_dict["Method"].append("POME")
        result_dict["Target"].append("Aplasia")
        result_dict["Score"].append(pome_auc)
        result_dict["Precision"].append(pome_aplasia_precision[i])
        result_dict["Recall"].append(pome_aplasia_recall[i])
        result_dict["F1"].append(pome_aplasia_f1[i])
        result_dict["Type"].append("AP")
        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("Aplasia")
        result_dict["Score"].append(umap_auc)
        result_dict["Precision"].append(umap_aplasia_precision[i])
        result_dict["Recall"].append(umap_aplasia_recall[i])
        result_dict["F1"].append(umap_aplasia_f1[i])
        result_dict["Type"].append("AP")
    
    result_dict["Method"].append("Naive")
    result_dict["Target"].append("Aplasia")
    result_dict["Score"].append(pos_class_ratio)
    result_dict["Precision"].append(-1.0)
    result_dict["Recall"].append(-1.0)
    result_dict["F1"].append(-1.0)
    result_dict["Type"].append("AP")
    
    # Separability analysis for Neutropenic fever.
    mimic_nf_df = mimic_df.copy()
    mimic_nf_y = mimic_nf_df["label_nf"].values
    mimic_umap_nf_df = mimic_umap.copy()
    pos_class_ratio = mimic_nf_y.mean()
    
    patients = np.array(list(patient_admission_dict.keys()))
    patients = shuffle(patients, random_state=42)

    n_folds = 10
    patient_folds = np.array_split(patients, n_folds)

    pome_nf_auc_scores = []
    pome_nf_precision = []
    pome_nf_recall = []
    pome_nf_f1 = []
    umap_nf_auc_scores = []
    umap_nf_precision = []
    umap_nf_recall = []
    umap_nf_f1 = []
    
    for fold_idx in range(n_folds):

        # Patients in test fold
        test_patients = patient_folds[fold_idx]

        # Patients in train folds
        train_patients = np.concatenate(
            [patient_folds[i] for i in range(n_folds) if i != fold_idx]
        )

        # Collect row indices
        train_rows = []
        for p in train_patients:
            train_rows.extend(patient_admission_dict[p])

        test_rows = []
        for p in test_patients:
            test_rows.extend(patient_admission_dict[p])

        # Extract dataframe rows
        pome_train_df = mimic_nf_df.iloc[train_rows]
        pome_test_df = mimic_nf_df.iloc[test_rows]
        
        umap_train_df = mimic_umap_nf_df.iloc[train_rows]
        umap_test_df = mimic_umap_nf_df.iloc[test_rows]

        # Compute POME AP.
        pome_X_train = pome_train_df.iloc[:, :NUM_DIMENSION].values
        pome_y_train = pome_train_df["label_nf"].values

        pome_X_test = pome_test_df.iloc[:, :NUM_DIMENSION].values
        pome_y_test = pome_test_df["label_nf"].values

        clf.fit(pome_X_train, pome_y_train)

        pome_y_pred = clf.predict_proba(pome_X_test)[:, 1]
        
        # Compute precision and recall.
        pome_y_pred_cut = (pome_y_pred >= 0.5).astype(int)
        pome_precision = precision_score(pome_y_test, pome_y_pred_cut)
        pome_recall = recall_score(pome_y_test, pome_y_pred_cut)
        pome_f1 = f1_score(pome_y_test, pome_y_pred_cut)
        pome_score = average_precision_score(pome_y_test, pome_y_pred)
        
        pome_nf_auc_scores.append(pome_score)
        pome_nf_precision.append(pome_precision)
        pome_nf_recall.append(pome_recall)
        pome_nf_f1.append(pome_f1)
        
        # Compute UMAP AP.
        assert not "label_nf" in umap_train_df.columns[:NUM_DIMENSION]
        umap_X_train = umap_train_df.iloc[:, :NUM_DIMENSION].values
        umap_y_train = umap_train_df["label_nf"].values

        assert not "label_nf" in umap_train_df.columns[:NUM_DIMENSION]
        umap_X_test = umap_test_df.iloc[:, :NUM_DIMENSION].values
        umap_y_test = umap_test_df["label_nf"].values

        clf.fit(umap_X_train, umap_y_train)

        umap_y_pred = clf.predict_proba(umap_X_test)[:, 1]

        # Compute additional precision and recall.
        umap_y_pred_cut = (umap_y_pred >= 0.5).astype(int)
        umap_precision = precision_score(umap_y_test, umap_y_pred_cut)
        umap_recall = recall_score(umap_y_test, umap_y_pred_cut)
        umap_f1 = f1_score(umap_y_test, umap_y_pred_cut)
        umap_score = average_precision_score(umap_y_test, umap_y_pred)
        
        umap_nf_auc_scores.append(umap_score)
        umap_nf_precision.append(umap_precision)
        umap_nf_recall.append(umap_recall)
        umap_nf_f1.append(umap_f1)
    
    assert len(pome_nf_auc_scores)==len(umap_nf_auc_scores)
    for i in range(len(pome_nf_auc_scores)):
        pome_auc = pome_nf_auc_scores[i]
        umap_auc = umap_nf_auc_scores[i]
        result_dict["Method"].append("POME")
        result_dict["Target"].append("Neutropenic Fever")
        result_dict["Score"].append(pome_auc)
        result_dict["Precision"].append(pome_nf_precision[i])
        result_dict["Recall"].append(pome_nf_recall[i])
        result_dict["F1"].append(pome_nf_f1[i])
        result_dict["Type"].append("AP")
        result_dict["Method"].append("UMAP")
        result_dict["Target"].append("Neutropenic Fever")
        result_dict["Score"].append(umap_auc)
        result_dict["Precision"].append(umap_nf_precision[i])
        result_dict["Recall"].append(umap_nf_recall[i])
        result_dict["F1"].append(umap_nf_f1[i])
        result_dict["Type"].append("AP")
    
    result_dict["Method"].append("Naive")
    result_dict["Target"].append("Neutropenic Fever")
    result_dict["Score"].append(pos_class_ratio)
    result_dict["Precision"].append(-1.0)
    result_dict["Recall"].append(-1.0)
    result_dict["F1"].append(-1.0)
    result_dict["Type"].append("AP")

/home/wollerf/miniforge3/envs/work/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wollerf/miniforge3/envs/work/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/wollerf/miniforge3/envs/work/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/

In [8]:
result_df = pd.DataFrame(result_dict)
result_df
result_df.to_csv(f"MIMIC_regression_results_{NUM_DIMENSION}.csv", index=False)